Simple RAG Demo — HR Policy Assistant
RAG stands for Retrieval-Augmented Generation. In plain English:

We take a document (here, an HR policy handbook).
We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
We store those pieces in a searchable database (a "vector store").
When a user asks a question, we search the database for the most relevant pieces, and hand them to an AI model to generate a final answer.
That's it. No magic — just search + a language model.

What we use in this notebook:

📄 Data: data/hr_policy.txt (a sample HR policy document)
🔢 Embeddings: Jina AI
🗄️ Vector store: FAISS
🧠 LLM: Groq (fast + free-tier friendly)
🕸️ Framework: LangChain (create_agent)


#Step 1 — Import everything we need
We import all the tools upfront so it's clear what's being used and where it comes from.

In [13]:
import os 
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings

C:\Users\RUPA\AppData\Local\Temp\ipykernel_2404\3242872321.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
load_dotenv()

python-dotenv could not parse statement starting at line 5


True

In [9]:
from dotenv import load_dotenv

load_dotenv()  # This loads the variables from your .env file

True

In [11]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")



#LOADING OUR DATA

In [12]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

#DATA INGESTION

In [14]:

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print("data loaded")
print(documents)

data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

#LANGCHAIN DOCUMENT
Langchain processes everything in form of documents

DOCUMENTS :

PAGE CONTENT -- the actual data

METADATA - extra information about the data

In [15]:
len(documents)

1

In [17]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [18]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [19]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2598


#SPLITTING OUR DATA

In [33]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [34]:
len(chunks)

9

#NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [35]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [36]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


In [37]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [38]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [39]:
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


#EMBEDD OUR DATA

In [40]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


#STORE DATA IN VECTOR DB

In [41]:
from langchain_community.vectorstores import FAISS 
vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [43]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



#TOOL

In [58]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

#DATA RETRIVAL
#LLM

In [44]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [53]:
question = llm.invoke("Hey what is machine learning")

In [54]:
print(question.content)

**Machine learning (ML)** is a branch of artificial intelligence (AI) that gives computers the ability to learn from data and improve their performance on a task without being explicitly programmed for every possible situation.

### How It Works (in a nutshell)

1. **Data Collection** – Gather examples (called *training data*) that show the relationship you care about (e.g., pictures of cats vs. dogs, past sales numbers, sensor readings, etc.).
2. **Model Choice** – Pick a mathematical structure (a *model*) that can capture patterns in the data. Common families include linear models, decision trees, neural networks, and support‑vector machines.
3. **Training** – Feed the data to the model and let an optimization algorithm adjust the model’s internal parameters so that its predictions match the known outcomes as closely as possible.
4. **Evaluation** – Test the trained model on new, unseen data to see how well it generalizes. Metrics like accuracy, precision, recall, or mean‑squared err

AI AGENT contain 3 things



LLM - BRAIN

TOOL - SUPER POWER

MEMORY - no memory (we awill not use here)

In [55]:
from langchain.agents import create_agent 

In [65]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for EY. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [66]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [67]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [68]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='42982e17-ec75-4aa7-b3cd-fd4b9a74a645'),
  AIMessage(content='I’m an HR assistant here at\u202fEY.', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is supposed to be a friendly HR assistant working for EY. We should answer that we work for EY. No need to search HR policy. The instruction says always use the search_hr_policy tool to look up facts before answering. But the question is about which org we work for; that is presumably known: EY. Could we search? Might not be in policy. But we can answer directly: "I work for EY". It\'s not a policy question. The developer instruction says always use the search_hr_policy tool to look up facts before answering. But if the answer isn\'t in the search results, say you don\'t know instead of guessing. However, the fact that we work for EY is not a policy fact; it\'s a stateme

SYSTEM MESSAGE - HR ASSISTANT

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THESE ARE THEPLOICES

In [69]:
response["messages"][-2].content

'tell me which org you work for'

In [70]:
print(response["messages"][-1].content)

I’m an HR assistant here at EY.
